In [7]:
import os, sys, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import Ridge

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

here = Path.cwd()
PROJECT_DIR = here.parent if here.name == 'notebooks' else here
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

def resolve_data_dir() -> Path:
    if 'DATA_DIR' in os.environ and Path(os.environ['DATA_DIR']).is_dir():
        return Path(os.environ['DATA_DIR'])
    for candidate in [PROJECT_DIR / 'data', here / 'data', Path.home() / 'Downloads']:
        if candidate.is_dir() and any(candidate.glob('*train*.csv')):
            return candidate
    return PROJECT_DIR / 'data'

DATA_DIR = resolve_data_dir()
TARGET = 'posted_rate'
EQUIPMENT = ['Dry Van', 'Flatbed', 'Reefer']
START_DATE = pd.Timestamp('2025-01-01')
DIST_BINS = [0, 150, 300, 500, 800, 1200, 2000, 5000]

def find(stem):
    candidates = [DATA_DIR, PROJECT_DIR / 'data', Path('data')]
    names = [stem, stem.replace('_', '-')]
    for d in candidates:
        if not d.is_dir():
            continue
        for n in names:
            if (d / f'{n}.csv').is_file():
                return str(d / f'{n}.csv')
        for n in names:
            hits = sorted(glob.glob(str(d / f'{n} (*).csv')))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'Could not find {stem}.csv in {DATA_DIR}.')

train = pd.read_csv(find('train_test'), parse_dates=['date'])
val = pd.read_csv(find('validation'), parse_dates=['date'])
dec = pd.read_csv(find('december_chart_inputs'), parse_dates=['date'])
print('train:', train.shape, '| val:', val.shape, '| dec:', dec.shape)


(48000, 14) (12000, 13) (31, 7)


In [8]:
class Cleaner:
    """Learns fill values from train, then applies them to any data."""

    def fit(self, df):
        weights = df["weight"].abs()
        self.weight_by_equipment = weights.groupby(df["equipment"]).median().to_dict()
        self.weight_overall = weights.median()
        return self

    def transform(self, df, market_by_day):
        out = df.copy()
        out["weight"] = out["weight"].abs()                                  # fix negatives
        out["weight"] = (out["weight"]
                         .fillna(out["equipment"].map(self.weight_by_equipment))
                         .fillna(self.weight_overall))                       # fix missing
        out["market_index"] = out["market_index"].fillna(out["date"].map(market_by_day))
        return out

# market_index is one number per day, so a day's median can fill its gaps.
# It uses features only, never labels, so there is no leakage.
market_by_day = pd.concat([train, val]).groupby("date")["market_index"].median()

cleaner = Cleaner().fit(train)
tr = cleaner.transform(train, market_by_day)

In [9]:
tr["dist_bin"] = pd.cut(tr["distance"], DIST_BINS)
tr["rate_per_mile"] = tr[TARGET] / tr["distance"]
typical = tr.groupby(["equipment", "dist_bin"])["rate_per_mile"].transform("median")
ratio = tr["rate_per_mile"] / typical
tr["is_corrupted"] = (ratio < 0.5) | (ratio > 2.0)
print("Corrupted rows:", tr["is_corrupted"].sum(), f"({tr['is_corrupted'].mean():.2%})")

Corrupted rows: 677 (1.41%)


In [10]:
def make_features(df):
    X = pd.DataFrame(index=df.index)
    X["log_dist"] = np.log(df["distance"])        # rate per mile falls with distance
    X["weight"] = df["weight"]
    X["equip"] = pd.Categorical(df["equipment"], categories=EQUIPMENT).codes
    X["market"] = df["market_index"]               # the daily market level
    X["t"] = (df["date"] - START_DATE).dt.days / 30.0   # months since Jan 1
    X["quote"] = df["quote_signal"]                # used only in the ablation
    return X

In [11]:
def score(y, p):
    err = np.abs(y - p)
    return {"MAE": err.mean(), "WAPE": err.sum() / y.sum() * 100}

class LookupBaseline:
    """Median $/mile for each equipment x distance band."""
    def fit(self, df):
        bins = pd.cut(df["distance"], DIST_BINS)
        self.table = (df[TARGET] / df["distance"]).groupby([df["equipment"], bins]).median()
        return self

    def predict(self, df):
        bins = pd.cut(df["distance"], DIST_BINS)
        keys = pd.MultiIndex.from_arrays([df["equipment"], bins])
        rpm = self.table.reindex(keys).fillna(self.table.median()).values
        return rpm * df["distance"].values

In [12]:
GBM_PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15,
                  min_child_samples=50, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, random_state=0, verbose=-1)
STATIC = ["log_dist", "weight", "equip"]

class PlainGBM:
    """One tree model on everything. Predicts log(rate)."""
    def __init__(self, cols):
        self.cols = cols
    def fit(self, df):
        self.model = lgb.LGBMRegressor(**GBM_PARAMS)
        self.model.fit(make_features(df)[self.cols], np.log(df[TARGET]))
        return self
    def predict(self, df):
        return np.exp(self.model.predict(make_features(df)[self.cols]))

class Hybrid:
    """Step 1: a simple linear model captures the market level (and optionally time drift).
       Step 2: a GBM learns what the line missed, from route features only."""
    def __init__(self, linear_cols, gbm_cols, alpha=1.0):
        self.linear_cols, self.gbm_cols, self.alpha = linear_cols, gbm_cols, alpha

    def _linear_X(self, df):
        X = make_features(df)
        out = X[self.linear_cols].copy()
        for code in (1, 2):                         # equipment dummies
            out[f"equip_{code}"] = (X["equip"] == code).astype(int)
        return out

    def fit(self, df):
        y = np.log(df[TARGET])
        X = self._linear_X(df)
        self.mean, self.std = X.mean(), X.std() + 1e-9
        self.linear = Ridge(alpha=self.alpha).fit((X - self.mean) / self.std, y)
        residual = y - self.linear.predict((X - self.mean) / self.std)
        self.gbm = lgb.LGBMRegressor(**GBM_PARAMS).fit(make_features(df)[self.gbm_cols], residual)
        return self

    def predict(self, df):
        X = (self._linear_X(df) - self.mean) / self.std
        log_pred = self.linear.predict(X) + self.gbm.predict(make_features(df)[self.gbm_cols])
        return np.exp(log_pred)

In [13]:
candidates = {
    "lookup":               lambda: LookupBaseline(),
    "plain_gbm":            lambda: PlainGBM(STATIC + ["market", "t"]),
    "hybrid_with_time":     lambda: Hybrid(["log_dist", "market", "t"], STATIC),
    "hybrid_no_time":       lambda: Hybrid(["log_dist", "market"], STATIC),
    "hybrid_no_time+quote": lambda: Hybrid(["log_dist", "market", "quote"], STATIC + ["quote"]),
}
folds = {"V1": ("2025-07-01", "2025-08-31"), "V2": ("2025-08-01", "2025-09-30"),
         "V3": ("2025-09-01", "2025-10-31"), "V4": ("2025-10-01", "2025-10-31")}

results = []
for fold, (start, end) in folds.items():
    train_part = tr[(tr["date"] < start) & ~tr["is_corrupted"]]
    test_part = tr[(tr["date"] >= start) & (tr["date"] <= end) & ~tr["is_corrupted"]]
    for name, build in candidates.items():
        pred = build().fit(train_part).predict(test_part)
        results.append({"fold": fold, "model": name, **score(test_part[TARGET].values, pred)})

results = pd.DataFrame(results)
print(results.pivot(index="model", columns="fold", values="WAPE").round(2))
print(results.groupby("model")[["MAE", "WAPE"]].mean().round(2))

fold                    V1    V2    V3    V4
model                                       
hybrid_no_time        2.63  4.18  4.53  3.28
hybrid_no_time+quote  2.85  4.18  4.43  3.45
hybrid_with_time      4.68  2.51  2.56  2.21
lookup                3.69  3.51  3.50  3.51
plain_gbm             4.51  2.78  2.98  3.03
                        MAE  WAPE
model                            
hybrid_no_time        85.85  3.65
hybrid_no_time+quote  87.56  3.73
hybrid_with_time      70.22  2.99
lookup                83.47  3.55
plain_gbm             78.12  3.33
